## Model Experiments
### In this notebook we perform experiments with different modelling & feature engineering techniques. We will compare with a baseline reference which is just the avg of the neighbourhood where the real estate is. 

#### Here is the flow of the notebook:
- Split into train/test group <br><br>
- Create lookup tables using the **train group data only**. These tables contain statistics (avg/std/counts) for the prices of groupings of our data. These groupings can be based on different citeria - neighbourhood, number of rooms, year of built, etc. We will bein with neighbourhood only stats for now because we have seen in previous experiments that this proves to be a major factor for price valuation. <br><br>
- Smoothen the values in these lookup tables. **Smoothening** is a technique which helps regularize the prediction. It works by telling the model how much to trust the value it sees (E.g. avg price) based on the number of observations for the specific group and if it is insufficient, it relies more on a more global statistics. **E.g.** If we have only 2 listings for a neighbourhood, we can't really trust that the avg price of the 2 is enough to give a general estimate for the entire region, so we fallback to the globl avg price more than the specific-region one. The formula is as follows:

$$\frac{n * \mu_{\text{group}} + m * \mu_{\text{global}}}{n + m}$$

where $m$ is a factor you determine (e.g. 10, 30, 50, etc.) <br> <br>

- Build model using only the averages and compute the errors. <br>
- Perform other experiments to compare... <br><br>
**Note**: If we pass a new unseen neigbourhood, we just pass the global avg as a baseline.

In [43]:
import sys, os

# backend/ — so `from model.src.x import ...` resolves via namespace packages
backend_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# backend/model/src/ — so bare imports inside src files (from config import ...) work
src_dir = os.path.join(backend_dir, "model", "src")

for p in [backend_dir, src_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)


In [45]:
import os, sys
print("cwd:", os.getcwd())
print("sys.path:", sys.path)

cwd: d:\dev\python\price-prediction-tool\backend\notebooks
sys.path: ['d:\\dev\\python\\price-prediction-tool\\backend\\model\\src', 'd:\\dev\\python\\price-prediction-tool\\backend', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64', 'd:\\venvs\\datasci', '', 'd:\\venvs\\datasci\\Lib\\site-packages', 'd:\\venvs\\datasci\\Lib\\site-packages\\win32', 'd:\\venvs\\datasci\\Lib\\site-packages\\win32\\lib', 'd:\\venvs\\datasci\\Lib\\site-packages\\Pythonwin']


In [46]:
import os, sys
print("cwd:", os.getcwd())
print("sys.path:", sys.path)

cwd: d:\dev\python\price-prediction-tool\backend\notebooks
sys.path: ['d:\\dev\\python\\price-prediction-tool\\backend\\model\\src', 'd:\\dev\\python\\price-prediction-tool\\backend', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'C:\\Users\\stoev\\AppData\\Local\\Python\\pythoncore-3.14-64', 'd:\\venvs\\datasci', '', 'd:\\venvs\\datasci\\Lib\\site-packages', 'd:\\venvs\\datasci\\Lib\\site-packages\\win32', 'd:\\venvs\\datasci\\Lib\\site-packages\\win32\\lib', 'd:\\venvs\\datasci\\Lib\\site-packages\\Pythonwin']


In [47]:
from model.src.data_load import load_data
from model.src.train import run_trial

In [48]:
data = load_data()
data.head()

🔌 Connecting to Neon...


,hash_id,title,img_url,link,neighbourhood,type_of_estate,total_price_eur,price_m2_eur,price_m2_bgn,size_m2,...,is_first_floor,is_last_floor,akt16,energy_class,potreblenie,broker_commision,additional_notes,is_furnished,near_public_transport,extras
0,191cff973bff3518de40c8019f6c3d5fea7174c95837aa...,"Имот - продава Двустаен апартамент, в София, М...",https://www.imoti.net/web/files/obiavi/6256545...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Малинова Долина,жилище,128991.00,2263.00,4426.32,57.00,...,False,True,True,N/A,N/A,True,Посочената цена не включва местни данъци и такси.,False,False,None
1,9eb41908d4e61b7008f29eb1688015f63ff3cdf229084b...,"Имот - продава Двустаен апартамент, в София, М...",https://www.imoti.net/web/files/obiavi/6256546...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Малинова Долина,жилище,128991.00,2263.00,4426.32,57.00,...,False,True,True,N/A,N/A,True,Посочената цена не включва местни данъци и такси.,False,False,None
2,dbc5d9466f8d880697f9fee292fe7c5fc84663ec10bcd4...,"Имот - продава Двустаен апартамент, в София, В...",https://www.imoti.net/web/files/obiavi/6262276...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Връбница 2,жилище,129000.00,2580.00,5046.04,50.00,...,False,False,True,N/A,N/A,True,Посочената цена не включва местни данъци и такси.,True,True,Обзаведен; ТЕЦ; Асансьор
3,6833ef4bed6c98041e1c7433fc60698103a43595c45950...,"Имот - продава Двустаен апартамент, в София, В...",https://www.imoti.net/web/files/obiavi/6246165...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Връбница 2,жилище,128984.00,2804.00,5484.83,46.00,...,False,False,True,N/A,N/A,True,Посочената цена не включва местни данъци и такси.,True,True,Обзаведен
4,65707b6e4a05c4b88ad12e0779c1fc35bd125f66133bc6...,"Имот - продава Едностаен апартамент, в София, ...",https://www.imoti.net/web/files/obiavi/6248616...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Малинова Долина,жилище,133014.00,3167.00,6193.46,42.00,...,False,False,True,N/A,N/A,True,Посочената цена не включва местни данъци и такси.,False,True,None


In [53]:
data["neighbourhood"].value_counts().reset_index().tail(50)

,neighbourhood,count
81,Люлин 7,9
82,Левски В,9
83,Младост 1а,9
84,Панчарево (с.),9
85,Обеля,8
86,Славия,8
87,Хиподрума,8
88,м-т Гърдова глава,7
89,Илинден,7
90,Американски колеж в.з.,7


In [55]:
def create_lookup_table(df, global_weight = 10):

    # convert price to numeric
    df["price_m2_eur"] = df["price_m2_eur"].astype(float)

    # calculte global avg/std
    global_avg = df["price_m2_eur"].mean()
    global_std = df["price_m2_eur"].std()

    # calculate the statistics
    lookup_table = df.groupby("neighbourhood")["price_m2_eur"].agg(
        count_neighbourhood="count",
        avg_price_neighbourhood="mean", 
        std_price_neighbourhood="std")

    # fill with the global std where we have neighbourhoods with only 1 observation
    lookup_table["avg_price_neighbourhood"] = lookup_table["avg_price_neighbourhood"].fillna(global_avg)
    lookup_table["std_price_neighbourhood"] = lookup_table["std_price_neighbourhood"].fillna(global_std)

    #smooth out the avg and the std
    lookup_table["smooth_avg_price_neighbourhood"] = (lookup_table["avg_price_neighbourhood"] * lookup_table["count_neighbourhood"] + global_std * global_weight) /( lookup_table["count_neighbourhood"] + global_weight)
    lookup_table["smooth_std_price_neighbourhood"] = (lookup_table["std_price_neighbourhood"] * lookup_table["count_neighbourhood"] + global_std * global_weight) /( lookup_table["count_neighbourhood"] + global_weight)

    return lookup_table.reset_index()

### Build Baseline

We want to all neighbourhoods that appear only once, to be part of the train set

In [58]:
from sklearn.model_selection import train_test_split
import pandas as pd

def stratified_split_with_singletons(df, stratify_col, test_size=0.2, random_state=42):
    '''Stratify on a specific column. This fucntion handles the case where we have only 1 observation of the group 
    and ads those to the training set'''
    # Drop rows where the stratify column is null — sklearn can't sort mixed str/None
    df = df[df[stratify_col].notna()].copy()

    # Count occurrences per category
    counts = df[stratify_col].value_counts()

    # Split into singletons and others
    singletons = df[df[stratify_col].isin(counts[counts == 1].index)]
    others = df[~df.index.isin(singletons.index)]

    # Stratified split on non-singletons
    train_other, test_other = train_test_split(
        others,
        test_size=test_size,
        stratify=others[stratify_col],
        random_state=random_state
    )

    # Add singletons ONLY to train
    train = pd.concat([train_other, singletons])
    test = test_other

    return train, test

train_group, test_group = stratified_split_with_singletons(data, "neighbourhood")
lookup_table = create_lookup_table(train_group)
train_group = pd.merge(train_group, lookup_table, on="neighbourhood", how="left")
test_group = pd.merge(test_group, lookup_table, on="neighbourhood", how="left")

In [62]:
import numpy as np

def run_baseline_model(test_data):
    test_data = test_data.copy()
    test_data["difference"] = test_data["smooth_avg_price_neighbourhood"] - test_data["price_m2_eur"]
    test_data["abs_error"] = test_data["difference"].abs()
    test_data["pct_error"] = (test_data["abs_error"] / test_data["price_m2_eur"]) * 100

    log_pred   = np.log(test_data["smooth_avg_price_neighbourhood"])
    log_actual = np.log(test_data["price_m2_eur"])
    log_diff   = log_pred - log_actual

    mae  = log_diff.abs().mean()
    rmse = np.sqrt((log_diff ** 2).mean())

    print(f"Baseline (smooth neighbourhood avg)")
    print(f"  MAE:  {mae:.2f} EUR/m²")
    print(f"  RMSE: {rmse:.2f} EUR/m²")

    return test_data

In [ ]:
import mlflow

mlflow.set_experiment(experiment_name="real-estate-price-per-m2-v1")

baseline_results = run_baseline_model(test_group)
baseline_results[["neighbourhood", "price_m2_eur", "smooth_avg_price_neighbourhood", "difference", "abs_error", "pct_error"]].head(20)

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from category_encoders import TargetEncoder
import numpy as np
import pandas as pd

from features import build_target

# ── Prepare data with log target ─────────────────────────────────────────────
df, target_transformed = build_target(data, "price_m2_eur")

BASE_FEATURES   = ["size_m2", "nr_of_rooms", "floor", "building_total_floors",
                   "neighbourhood", "is_first_floor", "is_last_floor", "is_furnished", "near_public_transport"]
LOOKUP_FEATURES = ["count", "smooth_avg_price_neighbourhood", "smooth_std_price_neighbourhood"]
ALL_FEATURES    = BASE_FEATURES + LOOKUP_FEATURES

_NUMERIC  = {"size_m2", "nr_of_rooms", "floor", "building_total_floors", "count", "smooth_std"}
_CAT      = {"neighbourhood"}
_BOOL     = {"is_first_floor", "is_last_floor", "is_furnished", "near_public_transport"}

def build_col_transformer_with_lookup(feature_cols):
    num   = [f for f in feature_cols if f in _NUMERIC]
    cat   = [f for f in feature_cols if f in _CAT]
    bool_ = [f for f in feature_cols if f in _BOOL]
    transformers = []
    if num:   transformers.append(("num",  Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                     ("scaler",  StandardScaler())]), num))
    if cat:   transformers.append(("cat",  TargetEncoder(), cat))
    if bool_: transformers.append(("bool", SimpleImputer(strategy="most_frequent"), bool_))
    return ColumnTransformer(transformers)

# ── Model configs (NOT dev mode) ─────────────────────────────────────────────
model_configs = [
    {
        "run_name": "XGBoost",
        "estimator": XGBRegressor(objective="reg:squarederror", random_state=42, nthread=1, tree_method="hist"),
        "param_grid": {
            "model__n_estimators":     [20, 40, 80, 160],
            "model__max_depth":        [10, 20, 40, 80],
            "model__learning_rate":    [0.1, 0.2, 0.4, 0.8],
            "model__colsample_bytree": [0.1, 0.2, 0.4, 0.8],
        },
    },
    {
        "run_name": "RandomForestRegressor",
        "estimator": RandomForestRegressor(random_state=42),
        "param_grid": {
            "model__n_estimators":    [80, 160, 320, 640],
            "model__max_depth":       [10, 20, 40, 80],
            "model__min_samples_leaf": [10, 20, 40, 80],
        },
    },
]

# ── Manual KFold CV loop ─────────────────────────────────────────────────────
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(df)):
    print(f"\n── Fold {fold_idx + 1} ──────────────────────────────────────────")

    train_fold = df.iloc[train_idx]
    test_fold  = df.iloc[test_idx]

    # Build lookup table from train fold only — prevents leakage
    lookup = create_lookup_table(train_fold)

    train_df = train_fold.merge(lookup, on="neighbourhood", how="left")
    test_df  = test_fold.merge(lookup,  on="neighbourhood", how="left")

    X_tr = train_df[ALL_FEATURES]
    y_tr = train_df[target_transformed]
    X_te = test_df[ALL_FEATURES]
    y_te = test_df[target_transformed]

    for cfg in model_configs:
        is_xgb = cfg["run_name"] == "XGBoost"

        # ── Step 1: GridSearchCV to find best hyperparams ─────────────────
        pipeline = Pipeline([
            ("preprocessor", build_col_transformer_with_lookup(ALL_FEATURES)),
            ("model", clone(cfg["estimator"])),
        ])
        gs = GridSearchCV(
            pipeline,
            cfg["param_grid"],
            cv=3,
            scoring="neg_root_mean_squared_error",
            refit=not is_xgb,   # RF: refit best directly; XGBoost: we refit with early stop
            n_jobs=-1,
            verbose=0,
        )
        gs.fit(X_tr, y_tr)
        best_params = {k.removeprefix("model__"): v for k, v in gs.best_params_.items()}

        if is_xgb:
            # ── Step 2 (XGBoost only): refit best params with early stopping ──
            X_tr_es, X_val_es, y_tr_es, y_val_es = train_test_split(
                X_tr, y_tr, test_size=0.1, random_state=42
            )
            preprocessor = build_col_transformer_with_lookup(ALL_FEATURES)
            X_tr_t  = preprocessor.fit_transform(X_tr_es, y_tr_es)
            X_val_t = preprocessor.transform(X_val_es)
            X_te_t  = preprocessor.transform(X_te)

            best_xgb = XGBRegressor(
                objective="reg:squarederror", random_state=42, nthread=1, tree_method="hist",
                early_stopping_rounds=20, eval_metric="rmse",
                **best_params,
            )
            best_xgb.fit(X_tr_t, y_tr_es, eval_set=[(X_val_t, y_val_es)], verbose=False)
            y_pred = best_xgb.predict(X_te_t)
        else:
            y_pred = gs.best_estimator_.predict(X_te)

        rmse = np.sqrt(np.mean((y_te.values - y_pred) ** 2))
        print(f"  {cfg['run_name']:30s}  RMSE={rmse:.4f}  best={best_params}")
        fold_results.append({"fold": fold_idx + 1, "model": cfg["run_name"], "rmse": rmse, **best_params})

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n── Summary ─────────────────────────────────────────────────────────")
results_df = pd.DataFrame(fold_results)
display(results_df.groupby("model")["rmse"].agg(["mean", "std"]).round(4))
results_df



── Fold 1 ──────────────────────────────────────────
657.5044813138088


C:\Users\stoev\AppData\Local\Temp\ipykernel_16308\2079243060.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price_m2_eur"] = df["price_m2_eur"].astype(float)


  XGBoost                         RMSE=0.1578  best={'colsample_bytree': 0.2, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 80}
  RandomForestRegressor           RMSE=0.1583  best={'max_depth': 20, 'min_samples_leaf': 10, 'n_estimators': 160}

── Fold 2 ──────────────────────────────────────────
647.0674016845375


C:\Users\stoev\AppData\Local\Temp\ipykernel_16308\2079243060.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price_m2_eur"] = df["price_m2_eur"].astype(float)


  XGBoost                         RMSE=0.1533  best={'colsample_bytree': 0.2, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 80}
  RandomForestRegressor           RMSE=0.1505  best={'max_depth': 20, 'min_samples_leaf': 10, 'n_estimators': 160}

── Fold 3 ──────────────────────────────────────────
649.5617733946992


...

  XGBoost                         RMSE=0.1491  best={'colsample_bytree': 0.4, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 80}
  RandomForestRegressor           RMSE=0.1528  best={'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 640}

── Fold 5 ──────────────────────────────────────────
652.8418322673095


C:\Users\stoev\AppData\Local\Temp\ipykernel_16308\2079243060.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price_m2_eur"] = df["price_m2_eur"].astype(float)


  XGBoost                         RMSE=0.1469  best={'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 40}
  RandomForestRegressor           RMSE=0.1531  best={'max_depth': 20, 'min_samples_leaf': 10, 'n_estimators': 80}

── Summary ─────────────────────────────────────────────────────────


,mean,std
model,,
RandomForestRegressor,0.1562,0.0064
XGBoost,0.1535,0.0057


,fold,model,rmse,colsample_bytree,learning_rate,max_depth,n_estimators,min_samples_leaf
0,1,XGBoost,0.157816,0.2,0.1,10,80,NaN
1,1,RandomForestRegressor,0.158277,NaN,NaN,20,160,10.0
2,2,XGBoost,0.153330,0.2,0.1,10,80,NaN
3,2,RandomForestRegressor,0.150501,NaN,NaN,20,160,10.0
4,3,XGBoost,0.160561,0.2,0.1,10,160,NaN
5,3,RandomForestRegressor,0.166440,NaN,NaN,20,320,10.0
6,4,XGBoost,0.149077,0.4,0.1,10,80,NaN
7,4,RandomForestRegressor,0.152751,NaN,NaN,10,640,10.0
8,5,XGBoost,0.146940,0.8,0.1,10,40,NaN
9,5,RandomForestRegressor,0.153100,NaN,NaN,20,80,10.0


In [42]:
import shap

# 1. Create a tree explainer using your trained model
explainer = shap.TreeExplainer(best_xgb)

# 2. Calculate SHAP values for your test or training features (X_test must be a pandas DataFrame or numpy array)
shap_values = explainer(X_tr)

# 3. Access the raw SHAP values array
print(shap_values.values)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:size_m2: object, neighbourhood: object, is_first_floor: object, is_last_floor: object

In [ ]:
from features import build_target

df, target_transformed = build_target(data, "price_m2_eur")

X_train, X_test, y_train, y_test = build_train_test_data(df,
                                                        target_col=target_transformed,
                                                        feature_set=FEATURE_SETS["all"])

col_transformer_pipeline = build_column_transformer_pipeline(feature_cols=f_set_values)